# TF-IDF baseline для общей классификации резюме

Ноутбук используется для обучения baseline-модели классификации резюме на основе `TF-IDF` и `LogisticRegression`.

## Импорты и настройка путей

In [11]:
from pathlib import Path

import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split


BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data" / "processed"
DATA_PATH = DATA_DIR / "resume_dataset.csv"

MODELS_DIR = BASE_DIR / "app" / "models"
VECTORIZER_PATH = MODELS_DIR / "tfidf_vectorizer.pkl"
CLASSIFIER_PATH = MODELS_DIR / "category_classifier.pkl"

TEXT_COLUMN = "resume_text"
TARGET_COLUMN = "target_role"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Dataset path:", DATA_PATH)
print("Models directory:", MODELS_DIR)


Base directory: /content
Dataset path: /content/data/processed/resume_dataset.csv
Models directory: /content/app/models


## Загрузка датасета

In [3]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (2481, 3)


,resume_text,target_role,source
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADM...,HR,huggingface_darshan_04_resume_classification
1,"HR SPECIALIST, US HR OPERATIONS Summary ...",HR,huggingface_darshan_04_resume_classification
2,HR DIRECTOR Summary Over 20 years e...,HR,huggingface_darshan_04_resume_classification
3,"HR SPECIALIST Summary Dedicated, Driv...",HR,huggingface_darshan_04_resume_classification
4,HR MANAGER Skill Highlights ...,HR,huggingface_darshan_04_resume_classification


## Базовая очистка данных

- оставляются только текст резюме и целевая категория;
- удаляются строки с пропусками;
- значения приводятся к строковому типу;
- удаляются слишком короткие тексты;
- удаляются дубликаты по паре `resume_text` и `target_role`.


In [4]:
df = df[[TEXT_COLUMN, TARGET_COLUMN]].dropna()

df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(str).str.strip()

df = df[df[TEXT_COLUMN].str.len() > 100]
df = df.drop_duplicates(subset=[TEXT_COLUMN, TARGET_COLUMN])

print("Dataset shape after cleaning:", df.shape)
print("\nClass distribution:")
df[TARGET_COLUMN].value_counts()


Dataset shape after cleaning: (2481, 2)

Class distribution:


,count
target_role,
INFORMATION-TECHNOLOGY,120
BUSINESS-DEVELOPMENT,119
ADVOCATE,118
CHEF,118
ENGINEERING,118
ACCOUNTANT,118
FINANCE,117
FITNESS,117
SALES,116


## Разделение на train и test

In [5]:
X = df[TEXT_COLUMN]
y = df[TARGET_COLUMN]

min_class_count = y.value_counts().min()
stratify = y if min_class_count >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=stratify,
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])


Train size: 1984
Test size: 497


## TF-IDF vectorizer и классификатор


In [6]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)

classifier = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

print(vectorizer)
print(classifier)

TfidfVectorizer(max_df=0.95, max_features=50000, min_df=2, ngram_range=(1, 2),
                stop_words='english')
LogisticRegression(class_weight='balanced', max_iter=3000, n_jobs=-1,
                   random_state=42)


## Обучение baseline-модели

In [7]:
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Train TF-IDF shape:", X_train_vec.shape)
print("Test TF-IDF shape:", X_test_vec.shape)

classifier.fit(X_train_vec, y_train)

Train TF-IDF shape: (1984, 50000)
Test TF-IDF shape: (497, 50000)


LogisticRegression(class_weight='balanced', max_iter=3000, n_jobs=-1,
                   random_state=42)

## Оценка качества

Используются метрики:

- `accuracy`
- `macro_f1`
- `weighted_f1`

In [12]:
y_pred = classifier.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")
precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)

metrics = {
    "model_name": "TF-IDF + LogisticRegression baseline",
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "precision_macro": precision_macro,
    "recall_macro": recall_macro,
}

print("Metrics:")
print(f"accuracy:        {accuracy:.4f}")
print(f"macro_f1:        {macro_f1:.4f}")
print(f"weighted_f1:     {weighted_f1:.4f}")
print(f"precision_macro: {precision_macro:.4f}")
print(f"recall_macro:    {recall_macro:.4f}")

metrics

Metrics:
accuracy:        0.6579
macro_f1:        0.6276
weighted_f1:     0.6448
precision_macro: 0.6702
recall_macro:    0.6300


{'model_name': 'TF-IDF + LogisticRegression baseline',
 'accuracy': 0.6579476861167002,
 'macro_f1': 0.62762024073853,
 'weighted_f1': 0.6448107972623766,
 'precision_macro': 0.6701567568234235,
 'recall_macro': 0.6300078163636517}

## Classification report


In [13]:
print(classification_report(y_test, y_pred, zero_division=0))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.56      0.83      0.67        24
              ADVOCATE       0.39      0.46      0.42        24
           AGRICULTURE       0.78      0.54      0.64        13
               APPAREL       0.50      0.16      0.24        19
                  ARTS       0.55      0.29      0.38        21
            AUTOMOBILE       0.75      0.43      0.55         7
              AVIATION       0.81      0.74      0.77        23
               BANKING       0.83      0.65      0.73        23
                   BPO       0.50      0.25      0.33         4
  BUSINESS-DEVELOPMENT       0.55      0.92      0.69        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.82      0.82      0.82        22
            CONSULTANT       0.50      0.17      0.26        23
              DESIGNER       0.86      0.86      0.86        21
         DIGITAL-MEDIA       0.77      

## Сохранение baseline-модели

После обучения сохраняются два артефакта:

- `tfidf_vectorizer.pkl`
- `category_classifier.pkl`


In [14]:
joblib.dump(vectorizer, VECTORIZER_PATH)
joblib.dump(classifier, CLASSIFIER_PATH)

print(f"Vectorizer saved to: {VECTORIZER_PATH}")
print(f"Classifier saved to: {CLASSIFIER_PATH}")

Vectorizer saved to: /content/app/models/tfidf_vectorizer.pkl
Classifier saved to: /content/app/models/category_classifier.pkl
